In [1]:
import sys
from pathlib import Path

# Add project root to Python path
sys.path.append(str(Path().resolve().parent))

# Test — HGT Encoder + DistMult Decoder

Tests each fix against the real graph object from Step 1/2 (`../graph/heterodata_with_features.pt`).

In [2]:
import torch
from models.encoder import HGTEncoder, build_message_passing_edges, split_polypharmacy_edges
from models.decoder import DistMultDecoder

PROJECT_ROOT = Path("..")

GRAPH_DIR = PROJECT_ROOT / "graph"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

data = torch.load(GRAPH_DIR / "heterodata_with_features.pt", weights_only=False)
print(data)


c:\Users\sth3ayush\Desktop\Hackathon\IIMS x Perceptron 2026\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HeteroData(
  drug={
    num_nodes=70,
    x=[70, 2048],
  },
  gene={ num_nodes=19083 },
  side_effect={
    num_nodes=5004,
    disease_class=[5004],
    num_disease_classes=1,
  },
  (gene, interacts, gene)={ edge_index=[2, 1431224] },
  (drug, targets, gene)={ edge_index=[2, 2522] },
  (gene, targeted_by, drug)={ edge_index=[2, 2522] },
  (drug, causes, side_effect)={ edge_index=[2, 20345] },
  (side_effect, caused_by, drug)={ edge_index=[2, 20345] },
  (drug, polypharmacy, drug)={
    edge_index=[2, 801630],
    edge_type=[801630],
  }
)


## Test 1 — basic forward pass, correct shapes

In [3]:
torch.manual_seed(0)
encoder = HGTEncoder(data)
embeddings = encoder(data)  # safe by default: excludes polypharmacy edges from message passing

assert embeddings["drug"].shape == (data["drug"].num_nodes, 128)
assert embeddings["gene"].shape == (data["gene"].num_nodes, 128)
assert embeddings["side_effect"].shape == (data["side_effect"].num_nodes, 128)

print("drug:", embeddings["drug"].shape)
print("gene:", embeddings["gene"].shape)
print("side_effect:", embeddings["side_effect"].shape)
print("PASS")


drug: torch.Size([70, 128])
gene: torch.Size([19083, 128])
side_effect: torch.Size([5004, 128])
PASS


## Test 2 — leakage fix

In [4]:
ddi_key = ("drug", "polypharmacy", "drug")
edge_index = data[ddi_key].edge_index
src0, dst0 = edge_index[0, 0].item(), edge_index[1, 0].item()
print(f"testing edge: drug {src0} -> drug {dst0}")

torch.manual_seed(1)
encoder = HGTEncoder(data)
encoder.eval()

with torch.no_grad():
    mp_edges_full = build_message_passing_edges(data)
    emb_full = encoder(data, edge_index_dict=mp_edges_full)
    src_emb_full = emb_full["drug"][src0].clone()
    dst_emb_full = emb_full["drug"][dst0].clone()

    # ablate the same edge as before -- but since polypharmacy edges were
    # never in mp_edges_full to begin with, this should now be a no-op
    mask = ~(((edge_index[0] == src0) & (edge_index[1] == dst0)) |
             ((edge_index[0] == dst0) & (edge_index[1] == src0)))
    data_ablated = data.clone()
    data_ablated[ddi_key].edge_index = edge_index[:, mask]

    mp_edges_ablated = build_message_passing_edges(data_ablated)
    emb_ablated = encoder(data_ablated, edge_index_dict=mp_edges_ablated)
    src_emb_ablated = emb_ablated["drug"][src0]
    dst_emb_ablated = emb_ablated["drug"][dst0]

diff_src = (src_emb_full - src_emb_ablated).abs().max().item()
diff_dst = (dst_emb_full - dst_emb_ablated).abs().max().item()
print(f"max embedding change for drug {src0}: {diff_src:.8f}")
print(f"max embedding change for drug {dst0}: {diff_dst:.8f}")

assert diff_src == 0.0 and diff_dst == 0.0, "leakage still present -- embeddings changed"
print("PASS -- embeddings identical, polypharmacy edges no longer leak into message passing")


testing edge: drug 56 -> drug 62
max embedding change for drug 56: 0.00000000
max embedding change for drug 62: 0.00000000
PASS -- embeddings identical, polypharmacy edges no longer leak into message passing


## Test 3 — hidden_dim / num_heads validation

In [5]:
try:
    HGTEncoder(data, hidden_dim=130, num_heads=4)
    print("FAIL -- no error raised")
except AssertionError as e:
    print("PASS -- raised clear AssertionError:")
    print(" ", e)


PASS -- raised clear AssertionError:
  hidden_dim (130) must be divisible by num_heads (4) -- HGTConv splits hidden_dim evenly across heads


## Test 4 — isolated node type raises a clear error instead of a bare KeyError

In [6]:
from torch_geometric.data import HeteroData

toy = HeteroData()
toy["drug"].num_nodes = 5
toy["drug"].x = torch.randn(5, 16)
toy["gene"].num_nodes = 3
toy["side_effect"].num_nodes = 2

toy["gene", "acts_on", "drug"].edge_index = torch.tensor([[0, 1, 2], [0, 1, 2]])
toy["drug", "causes", "side_effect"].edge_index = torch.tensor([[0, 1], [0, 1]])
toy["side_effect", "caused_by", "drug"].edge_index = torch.tensor([[0, 1], [0, 1]])
# note: no edge type has "gene" as a destination

edge_index_dict = {et: toy[et].edge_index for et in toy.edge_types}

try:
    toy_encoder = HGTEncoder(toy, hidden_dim=16, num_heads=2, num_layers=2)
    toy_encoder(toy, edge_index_dict=edge_index_dict)
    print("FAIL -- no error raised")
except ValueError as e:
    print("PASS -- raised clear ValueError instead of a bare KeyError:")
    print(" ", e)


PASS -- raised clear ValueError instead of a bare KeyError:
  After conv layer 0, node type(s) {'gene'} produced no output -- they never appear as the destination of any edge type reachable from the current edge_index_dict. Add at least one incoming edge type for these node types, or exclude them from this encoder's node types.


## Test 5 — decoder relation-count validation

In [7]:
decoder = DistMultDecoder(num_relations=4, hidden_dim=128)

drug_embeddings = torch.randn(8, 128)
edge_index = torch.tensor([[0, 1], [3, 4]])
edge_type = torch.tensor([0, 5])  # relation id 5 doesn't exist in this decoder

try:
    decoder(drug_embeddings, edge_index, edge_type)
    print("FAIL -- no error raised")
except AssertionError as e:
    print("PASS -- raised clear AssertionError:")
    print(" ", e)


PASS -- raised clear AssertionError:
  edge_type contains relation id 5, but this decoder was constructed with num_relations=4. num_relations must exactly match len(relation_vocab.json) from Step 1 -- use DistMultDecoder.from_relation_vocab(...) to avoid this drifting out of sync.


## Test 6 — decoder constructed straight from the real relation lookup

In [8]:
relation_vocab_path = PROCESSED_DIR / "polypharmacy_relation_lookup.csv"

decoder = DistMultDecoder.from_relation_vocab(relation_vocab_path)
print("num_relations:", decoder.num_relations)


num_relations: 1301


## Test 7 — end-to-end: disjoint splits -> encoder (train edges only) -> decoder scores held-out edges

In [9]:
torch.manual_seed(2)
splits = split_polypharmacy_edges(data, val_frac=0.1, test_frac=0.1, seed=42)

print("train edges:", splits["train"]["edge_index"].shape[1])
print("val edges:  ", splits["val"]["edge_index"].shape[1])
print("test edges: ", splits["test"]["edge_index"].shape[1])

# confirm the three splits are actually disjoint -- at the (src, dst, relation)
# TRIPLE level, not just the (src, dst) pair. Multiple different relations
# can legitimately exist between the same drug pair (e.g. two drugs can share
# both "atelectasis" and "retinopathy" edges); splitting is meant to happen
# per triple, so seeing one relation for a pair in train and a DIFFERENT
# relation for that same pair in val/test is expected, not a leak.
def edge_set(split):
    ei = split["edge_index"]
    et = split["edge_type"]
    return set(zip(ei[0].tolist(), ei[1].tolist(), et.tolist()))

train_set, val_set, test_set = edge_set(splits["train"]), edge_set(splits["val"]), edge_set(splits["test"])
assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)
print("PASS -- train/val/test polypharmacy splits are disjoint")

# message passing uses every non-polypharmacy edge type + only the TRAIN
# slice of polypharmacy edges (mirrors how you'd actually train this)
mp_edges = build_message_passing_edges(data)
mp_edges[("drug", "polypharmacy", "drug")] = splits["train"]["edge_index"]

encoder = HGTEncoder(data)
embeddings = encoder(data, edge_index_dict=mp_edges)

num_relations = int(data[("drug", "polypharmacy", "drug")].edge_type.max().item()) + 1
decoder = DistMultDecoder(num_relations=num_relations, hidden_dim=128)

val_scores = decoder(embeddings["drug"], splits["val"]["edge_index"], splits["val"]["edge_type"])
print("val scores shape:", val_scores.shape)
assert val_scores.shape[0] == splits["val"]["edge_index"].shape[1]
print("PASS -- decoder scores held-out val edges using embeddings that never saw them during message passing")


train edges: 641304
val edges:   80163
test edges:  80163
PASS -- train/val/test polypharmacy splits are disjoint
val scores shape: torch.Size([80163])
PASS -- decoder scores held-out val edges using embeddings that never saw them during message passing
